# Sprint 3 — Comparación y selección de candidatos

    Selecciona candidatos para tuning. La selección es reproducible y queda guardada.

In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import *
from src.io_utils import load_kick_data, save_df
from src.preprocessing import prepare_features, split_X_y

pd.set_option('display.max_columns', 120)
print('Proyecto:', PROJECT_ROOT)

Proyecto: /Users/alexandralozano/dp261-g1


In [5]:
results = pd.read_csv(REPORTS_DIR / "baseline_cv_results.csv")

ok = results[
    results["status"].eq("ok") &
    ~results["model"].str.contains("Dummy", case=False, na=False)
].copy()

candidates = (
    ok
    .sort_values(
        ["recall_cv_mean", "f2_cv_mean", "precision_cv_mean"],
        ascending=[False, False, False]
    )
    .head(3)
    .copy()
)

candidates["selected_for_tuning"] = True

display(candidates[[
    "model",
    "recall_cv_mean",
    "f2_cv_mean",
    "precision_cv_mean",
    "f1_cv_mean",
    "f05_cv_mean",
    "roc_auc_cv_mean",
    "average_precision_cv_mean",
    "fit_time_mean",
    "selected_for_tuning"
]])

candidates.to_csv(
    REPORTS_DIR / "model_selection_candidates.csv",
    index=False
)

,model,recall_cv_mean,f2_cv_mean,precision_cv_mean,f1_cv_mean,f05_cv_mean,roc_auc_cv_mean,average_precision_cv_mean,fit_time_mean,selected_for_tuning
0,DecisionTree,0.347154,0.337769,0.304994,0.324648,0.312550,0.618047,0.186297,1.031457,True
1,LinearSVM,0.261382,0.299204,0.712861,0.382222,0.529340,0.755643,0.446534,0.447546,True
2,LogisticRegression,0.250813,0.288854,0.737390,0.373998,0.530694,0.757379,0.447457,0.358581,True


In [6]:
discard = ok[~ok["model"].isin(candidates["model"])].copy()

f2_median = ok["f2_cv_mean"].median()
recall_median = ok["recall_cv_mean"].median()

discard["discard_reason"] = np.select(
    [
        discard["recall_cv_mean"].fillna(0) == 0,
        discard["f2_cv_mean"].fillna(0) < f2_median,
        discard["recall_cv_mean"].fillna(0) < recall_median,
    ],
    [
        "no detecta Bad Buys o recall nulo",
        "menor F2 que los modelos priorizados",
        "menor recall que los modelos priorizados",
    ],
    default="menor prioridad para tuning"
)

display(discard[[
    "model",
    "recall_cv_mean",
    "f2_cv_mean",
    "precision_cv_mean",
    "f1_cv_mean",
    "f05_cv_mean",
    "discard_reason"
]])

discard.to_csv(
    REPORTS_DIR / "model_discard_reasons.csv",
    index=False
)

,model,recall_cv_mean,f2_cv_mean,precision_cv_mean,f1_cv_mean,f05_cv_mean,discard_reason
3,GradientBoosting,0.246748,0.287131,0.832756,0.380578,0.564339,menor prioridad para tuning
4,HistGradientBoosting,0.245935,0.286951,0.863707,0.382712,0.574578,menor F2 que los modelos priorizados
5,RandomForest,0.230488,0.270392,0.880477,0.365257,0.562769,menor F2 que los modelos priorizados
6,KNN,0.011789,0.014656,0.547273,0.023074,0.054220,menor F2 que los modelos priorizados
